<div align="left" style="background-color: #008080; padding: 20px 10px;">
<h3><b>IDEAS - Institute of Data Engineering, Analytics and Science Foundation</b></h3>
<p>Summer Internship Program 2026</p>
<hr style="width:100%;">
<h3><b>Project Title:</b> House Price India — Data Analysis</h3>
<h4>Project Notebook</h4>
<blockquote style="border-left: 4px solid #4285F4; padding-left: 15px;">
  <strong>Created by:</strong> Samyabrata Roy<br>
  <strong>Designation:</strong> Associate Software Developer
</blockquote>
<hr style="width:100%;">
</div>

### Question 1: Import Libraries and Load Data (2 Marks)

Import `pandas` as `pd` and `numpy` as `np`. Load the `house_price_india.csv` dataset into a DataFrame named `df` and display its first 5 rows.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('house_price_india.csv')
df.head()

### Question 2: Convert Date and Check Data Types (1 Mark)

The `Date` column is stored as an Excel serial number (integer). Convert it to a proper `datetime` format using the Excel epoch (`origin='1899-12-30'`). Then print the data types of all columns to confirm the change.

**Expected Output:** A Series listing each column and its data type, where `Date` is `datetime64`.

In [ ]:
# Convert Excel serial number to proper datetime
df['Date'] = pd.to_datetime(df['Date'], origin='1899-12-30', unit='D')
print(df.dtypes)

### Question 3: Data Inspection (1 Mark)

Get a quick overview of the DataFrame. Print its shape and check for the total number of missing (null) values across the entire DataFrame.

**Expected Output:** The shape of the DataFrame and the total count of null values.

In [ ]:
print("Shape:", df.shape)
print("Total null values:", df.isnull().sum().sum())

### Question 4: Monthly Revenue (Total Price) Calculation (2 Marks)

Calculate the total house sale value (`Price`) for each month. The result should be a Series indexed by month-year period. Display the result.

**Expected Output:** A Series with month-year periods as the index and total sale value as values.

In [ ]:
monthly_revenue = df.groupby(df['Date'].dt.to_period('M'))['Price'].sum()
print(monthly_revenue)

### Question 5: Weekly Revenue (Total Price) Calculation (1 Mark)

Similar to the previous question, calculate the total sale value (`Price`) for each week. Display the first 5 rows of the resulting Series.

**Expected Output:** The first 5 rows of a Series with week periods as the index and total sale value as values.

In [ ]:
weekly_revenue = df.groupby(df['Date'].dt.to_period('W'))['Price'].sum()
print(weekly_revenue.head())

### Question 6: Best-Performing House Grade (2 Marks)

Determine which `grade of the house` has generated the highest total sale value. Group the data by `grade of the house`, sum the `Price`, and find the grade with the maximum sum.

**Expected Output:** The best-performing grade and its total sale value.

In [ ]:
grade_revenue = df.groupby('grade of the house')['Price'].sum()
best_grade = grade_revenue.idxmax()
print(f"Best-Performing Grade: {best_grade}")
print(f"Total Sale Value: {grade_revenue[best_grade]:,}")

### Question 7: Top 5 Most Expensive Houses (2 Marks)

Identify the top 5 most expensive houses sold. Sort the data by `Price` in descending order and display the top 5 entries showing their `id` and `Price`.

**Expected Output:** A DataFrame showing the top 5 house IDs and their sale prices.

In [ ]:
top5_houses = df.nlargest(5, 'Price')[['id', 'Price']]
print(top5_houses)

### Question 8: Average Price per Number of Bedrooms (2 Marks)

Calculate the average house price for each bedroom count. Group by `number of bedrooms` and find the mean `Price`. Display the top 5 bedroom counts by average price.

**Expected Output:** A Series showing the top 5 bedroom counts by average sale price.

In [ ]:
avg_price_by_bedrooms = df.groupby('number of bedrooms')['Price'].mean().nlargest(5)
print(avg_price_by_bedrooms)

### Question 9: Cumulative Revenue Calculation (2 Marks)

Calculate the cumulative total sale value over time. First group the data by `Date` and sum the `Price`, then compute the cumulative sum. Display the last 5 entries.

**Expected Output:** A Series showing the last 5 dates and the cumulative total sale value up to each date.

In [ ]:
daily_revenue = df.groupby('Date')['Price'].sum()
cumulative_revenue = daily_revenue.cumsum()
print(cumulative_revenue.tail())

### Question 10: Pivot Table — Grade vs. Month (2 Marks)

Create a pivot table showing total `Price` for each `grade of the house` broken down by month. First extract the month number into a `Month` column, then build the pivot table with `Month` as rows and `grade of the house` as columns.

**Expected Output:** A table with months as rows, house grades as columns, and total sale value at each intersection.

In [ ]:
df['Month'] = df['Date'].dt.month
pivot = pd.pivot_table(
    df,
    values='Price',
    index='Month',
    columns='grade of the house',
    aggfunc='sum'
)
print(pivot)

### Question 11: Prepare Features for Clustering (3 Marks)

Select key numeric features from the house price dataset and prepare them as the feature matrix `X`. Also create a target label array `y` by binning `Price` into 5 equal-frequency tiers (0–4) using `pd.qcut`. Print the shapes of `X` and `y`.

**Expected Output:** The shape of `X` and `y`.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Select key numeric features
features = [
    'number of bedrooms', 'number of bathrooms', 'living area', 'lot area',
    'number of floors', 'grade of the house', 'condition of the house',
    'Area of the house(excluding basement)', 'Area of the basement',
    'Number of schools nearby', 'Distance from the airport'
]

X = df[features].values

# Create price tier labels: 5 equal-frequency bins (0 = cheapest, 4 = most expensive)
df['price_tier'] = pd.qcut(df['Price'], q=5, labels=[0, 1, 2, 3, 4]).astype(int)
y = df['price_tier'].values

print("Shape of X (features):", X.shape)
print("Shape of y (price tiers):", y.shape)

### Question 12: K-Means Clustering (5 Marks)

Import `KMeans` from `sklearn.cluster`. Scale the features using `StandardScaler`, then fit a K-Means model with 5 clusters (matching the 5 price tiers). Assign the cluster labels to `kmeans_labels`.

**Expected Output:** A successfully fitted K-Means model and the cluster labels array.

In [ ]:
from sklearn.cluster import KMeans

# Scale features for better clustering performance
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit K-Means with 5 clusters (one per price tier)
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
kmeans.fit(X_scaled)
kmeans_labels = kmeans.labels_

print("K-Means model fitted successfully!")
print("Cluster labels (first 20):", kmeans_labels[:20])

### Question 13: F1 Score Evaluation for Clustering (5 Marks)

Evaluate clustering performance using the F1 score. Since K-Means assigns arbitrary cluster IDs, first map each cluster to the most frequent true price tier it contains using `scipy.stats.mode`. Then compute and print the macro-averaged F1 score.

**Expected Output:** The macro-averaged F1 score of the clustering.

In [ ]:
from sklearn.metrics import f1_score
from scipy.stats import mode

# Map each cluster label to the most frequent true price tier in that cluster
mapped_labels = np.zeros_like(kmeans_labels)
for cluster in range(5):
    mask = kmeans_labels == cluster
    most_common = mode(y[mask], keepdims=True).mode[0]
    mapped_labels[mask] = most_common

# Calculate macro-averaged F1 score
f1 = f1_score(y, mapped_labels, average='macro')
print(f"Macro-averaged F1 Score: {f1:.4f}")